In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import plot_period_transactions
from lifetimes.utils import summary_data_from_transaction_data


file_path = "online_retail_II.xlsx"   

df_0910 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_1011 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

df = pd.concat([df_0910, df_1011], ignore_index=True)

print(f"Total rows before cleaning: {len(df)}")


df.columns = df.columns.str.strip() 

df = df[df['Country'] == 'United Kingdom'].copy()

df = df[df['Customer ID'].notna()].copy()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df = df[(df['Quantity'] > 0) & (df['Price'] > 0)].copy()

df['TotalPrice'] = df['Quantity'] * df['Price']

print(f"Rows after cleaning: {len(df)}")
print(df.head())

snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"Snapshot date for Recency: {snapshot_date}")


rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  
    'Invoice': 'nunique',                                     
    'TotalPrice': 'sum'                                       
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f"Number of unique customers: {len(rfm)}")
print(rfm.head())


rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])  
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] >= 2 and row['F_Score'] >= 2 and row['M_Score'] >= 2:
        return 'Potential'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

clv_df = df[['Customer ID', 'InvoiceDate', 'TotalPrice']].copy()
clv_df.columns = ['CustomerID', 'InvoiceDate', 'TotalPrice']

summary = summary_data_from_transaction_data(
    clv_df,
    'CustomerID',
    'InvoiceDate',
    'TotalPrice',
    observation_period_end=snapshot_date
)

print("\nSummary data before filtering:")
print(f"Total customers in summary: {len(summary)}")
print(summary.head())

summary_filtered = summary[(summary['frequency'] > 0) & (summary['monetary_value'] > 0)].copy()

print(f"\nCustomers with positive frequency and monetary value: {len(summary_filtered)}")
print(f"Customers removed: {len(summary) - len(summary_filtered)}")

bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(summary_filtered['frequency'], summary_filtered['recency'], summary_filtered['T'])
print("\nBG/NBD fitted successfully!")

summary_filtered['predicted_purchases'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    30, summary_filtered['frequency'], summary_filtered['recency'], summary_filtered['T']
)

ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(summary_filtered['frequency'], summary_filtered['monetary_value'])
print("Gamma‑Gamma fitted successfully!")

summary_filtered['predicted_avg_monetary'] = ggf.conditional_expected_average_profit(
    summary_filtered['frequency'], summary_filtered['monetary_value']
)

Total rows before cleaning: 1067371
Rows after cleaning: 725250
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  TotalPrice  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom        83.4  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom        81.0  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom        81.0  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom       100.8  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom        30.0  
Snapshot date for Recency: 2011-12-10 12:49:00
Number of unique customers: 5350
   

In [ ]:
summary_filtered['CLV'] = ggf.customer_lifetime_value(
    bgf,
    summary_filtered['frequency'],
    summary_filtered['recency'],
    summary_filtered['T'],
    summary_filtered['monetary_value'],
    time=12,
    discount_rate=0.01,
    freq='W'         
)

In [ ]:
print("\nCalculating CLV manually...")

summary_filtered['expected_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    52, 
    summary_filtered['frequency'],
    summary_filtered['recency'],
    summary_filtered['T']
)

summary_filtered['CLV'] = summary_filtered['expected_purchases_12m'] * summary_filtered['monetary_value']

print("Manual CLV calculated successfully.")
print(summary_filtered[['CLV']].head())

summary_all = summary.copy()
summary_all['CLV'] = 0
summary_all.loc[summary_filtered.index, 'CLV'] = summary_filtered['CLV']

rfm = rfm.merge(summary_all[['CLV']], left_on='CustomerID', right_index=True, how='left')
rfm['CLV'] = rfm['CLV'].fillna(0)

rfm.to_csv("rfm_clv_output.csv", index=False)
print("✅ Output saved to 'rfm_clv_output.csv'")


Calculating CLV manually...
Manual CLV calculated successfully.
                    CLV
CustomerID             
12346.0     2843.256318
12745.0       15.738934
12747.0      580.516740
12748.0     3493.614700
12749.0      616.712209
✅ Output saved to 'rfm_clv_output.csv'
